# LUMOS: Gavin's Internship Notebook

Gavin's personal jupyter notebook for reading metadata, data preprocessing, and finally photometric analysis of *T CrB*. Now with [LUMOS](https://github.com/gavinathaya/LUMOS)!

## Import Packages

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import lumos.io as lumio
import lumos.calibration as luca
import lumos.photometry as lupho
import lumos.visualization as luvi
from lumos.photometry import PhotometrySession
from astropy.coordinates import SkyCoord
from astropy import units as u
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd

## Reading the data

### Finding valid files

Using the following command on terminal (macOS/Linux):

    find ./Fotometri_Timau_2024/ -name "*TCrb*" -maxdepth 2 ! -name "*._*" ! -name "*dark*"

We will get the following results:

```
./Fotometri_Timau_2024//180524/180524-TCrb--002-45s-B.fit
./Fotometri_Timau_2024//180524/180524-TCrb--002-30s-R.fit
./Fotometri_Timau_2024//180524/180524-TCrb--001-30s-R.fit
./Fotometri_Timau_2024//180524/180524-TCrb--003-30s-V.fit
./Fotometri_Timau_2024//180524/180524-TCrb--004-45s-B.fit
./Fotometri_Timau_2024//180524/180524-TCrb--003-45s-B.fit
./Fotometri_Timau_2024//180524/180524-TCrb--003-30s-R.fit
./Fotometri_Timau_2024//180524/180524-TCrb--001-45s-B.fit
./Fotometri_Timau_2024//180524/180524-TCrb--002-30s-V.fit
./Fotometri_Timau_2024//180524/180524-TCrb--001-30s-V.fit
./Fotometri_Timau_2024//180524/180524-TCrb--005-45s-B.fit

```

Alternatively, we can use lumos's io module to do the same thing without using external programs.

In [ ]:
raw_list, dark_list, bias_list, flat_list = lumio.findfiles(dir = '/Volumes/labt1_share/gavin/.brin/Fotometri_Timau_2024/**/*.fit',
                                                           raw_name = "*TCrb*.fit",
                                                           dark_name = "*TCrb*dark.fit",
                                                           bias_name = "*bias.fit",
                                                           flat_name = "*flat.fit")

print(f"Found {len(raw_list)} valid files related to v CrB:\n{raw_list}"+
      f"{'\n' * 2}Found {len(dark_list)} valid dark calibration files related to T CrB:\n{dark_list}"+
      f"{'\n' * 2}Found {len(bias_list)} valid bias calibration files:\n{bias_list}"+
      f"{'\n' * 2}Found {len(flat_list)} valid flat calibrationx files:\n{flat_list}")

Looks like we got a stray NGC calibration file!

<p align="justify">
This is because it's bad practice to put different calibration file from different subjects. Though not to worry, we can simply filter it back by np.find.char()
<p>

In [ ]:
bias_list = bias_list[~(np.char.find(bias_list, 'NGC') != -1)]
print(f"{'\n' * 2}Found {len(bias_list)} valid bias calibration files:\n{bias_list}")

### Metadata Extraction & Image Preprocessing

<p align="justify">
In this section, we will leverage lumos.calibration.core's powerful CalibrationFrames class to preprocess the calibration frames in conjunction with lumos.io.metadata_gen's ability to create the necessary metadata in one fell swoop.
<p>

In [ ]:
#Generate metadata and CalibrationFrames object
metadata = lumio.metadata_gen(raw_list)
cal = luca.CalibrationFrames(metadata = metadata)

#Load calibration frames
cal.load_bias(bias_list)
cal.load_darks(dark_list)
cal.load_flats(flat_list)

## Calibration

As you will see, calibration is as easy as a line of python script.

The rest is handled by the object.

In [ ]:
cal.apply_self("/Volumes/labt1_share/gavin/.brin/results/calibrated_FITS/",
               "TCrB",
                "/Volumes/labt1_share/gavin/.brin/results/", warn = False)

and making - saving comparison images are *just* as easy

In [ ]:
cal.plot_calibration("/Volumes/labt1_share/gavin/.brin/results/calibration_plots/")

## Background Removal & Plotting

<p align="justify">
The calibrated image we just processed is free from any defects and/or quirks from the CCD Camera. Though the background is still present and we need to remove it before we can do any photometric analysis.
</p>

In [ ]:
cal.remove_background_self("/Volumes/labt1_share/gavin/.brin/results/cleaned_FITS/",
                           "TCrB", "/Volumes/labt1_share/gavin/.brin/results/", warn = False)

and plotting the background comparison images

In [ ]:
cal.plot_background("/Volumes/labt1_share/gavin/.brin/results/background_plots/")

## Photometric Analysis

### Star Identification

First, we use the DAOFIND algorithm through the DAOStarFinder object.   
In LUMOS, we can do this process for all of the frames using the PhotometrySession object

We will create the PhotometrySession object and process its ref_stars (catalogue)   
attribute to be filled with necessary data

In [ ]:
tcrb_ra = 239.87567583333333
tcrb_dec = 25.92017027777778
catalogue = pd.read_csv("/Volumes/labt1_share/gavin/.brin/results/table.csv")
catalogue = catalogue[['id', 'ra', 'dec', 'mag_b', 'mag_v', 'mag_b','mag_r']].dropna()
catalogue['distance'] = np.sqrt((catalogue['ra'] - tcrb_ra)**2 + (catalogue['dec'] - tcrb_dec)**2)
catalogue = catalogue.sort_values('distance')
metadata = pd.read_csv('/Volumes/labt1_share/gavin/.brin/results/TCrB_metadata.csv')

In [ ]:
photsesh = PhotometrySession(metadata = metadata,
                             ref_stars = catalogue,
                             )
photsesh.add_lightcurves("Blaze Star",SkyCoord(tcrb_ra * u.deg, tcrb_dec * u.deg)) # pyright: ignore[reportAttributeAccessIssue]
photsesh.ref_image = photsesh.wcs_files[0]

In [ ]:
photsesh.find_source(detection_dir = '/Volumes/labt1_share/gavin/.brin/results/detection_dir/',
                     subject_name="TCrB",
                     metadata_dir='/Volumes/labt1_share/gavin/.brin/results/')

In [ ]:
photsesh.plot_sources('/Volumes/labt1_share/gavin/.brin/results/source_plots/')

In [ ]:
photsesh.apply_aperture_photometry(phot_dir='/Volumes/labt1_share/gavin/.brin/results/phot_dir/',
                                  subject_name="TCrB",
                                  metadata_dir='/Volumes/labt1_share/gavin/.brin/results/')

Now let's see the lightcurve's state and its data

In [ ]:
photsesh.lightcurves

and the TimeSeries itself:

In [ ]:
photsesh.lightcurves['Blaze Star']['lightcurve']

and plotting the lightcurve itself

In [ ]:
times=photsesh.lightcurves['Blaze Star']['lightcurve'].time.jd
mag_use = photsesh.lightcurves['Blaze Star']['lightcurve']['mag_R']
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(times, mag_use,'k.', markersize=8)
ax.set(xlabel='time', ylabel='R Magnitude', title='Lightcurve of T CrB (Blaze Star)')

In [ ]:
from scipy.stats import median_abs_deviation

def get_clean_mag(ts, filter_name):
    col = f"mag_{filter_name}"

    mag = ts[col]
    time = ts.time

    # Handle masked + NaN
    valid = (~mag.mask) & np.isfinite(mag)

    return time[valid], mag[valid]

def plot_lightcurve(ts, filter_name):
    time, mag = get_clean_mag(ts, filter_name)

    plt.figure()
    plt.scatter(time.jd, mag, s=20)
    plt.gca().invert_yaxis()  # astronomy convention
    plt.xlabel("Time (JD)")
    plt.ylabel(f"{filter_name}-band magnitude")
    plt.title(f"TCrB - {filter_name} lightcurve")
    plt.show()

def plot_mag_histogram(ts, filter_name, bins='auto'):
    _, mag = get_clean_mag(ts, filter_name)

    plt.figure()
    plt.hist(mag, bins=bins)
    plt.xlabel(f"{filter_name}-band magnitude")
    plt.ylabel("Counts")
    plt.title(f"TCrB - {filter_name} magnitude distribution")
    plt.gca().invert_xaxis()
    plt.show()

def mag_stats(ts, filter_name):
    _, mag = get_clean_mag(ts, filter_name)

    mean = np.mean(mag)
    std = np.std(mag, ddof=1)

    median = np.median(mag)
    mad = median_abs_deviation(mag, scale='normal')

    return {
        "mean": mean,
        "std": std,
        "median": median,
        "mad": mad,
        "N": len(mag)
    }

def summary_table(ts, filters):
    rows = []
    for f in filters:
        s = mag_stats(ts, f)
        rows.append({
            "filter": f,
            "N": s["N"],
            "mean_mag": s["mean"],
            "std_mag": s["std"],
            "median_mag": s["median"],
            "mad_mag": s["mad"]
        })
    return pd.DataFrame(rows)

In [ ]:
filters = ["B", "V", "R"]
ts = photsesh.lightcurves['Blaze Star']['lightcurve']

for f in filters:
    plot_lightcurve(ts, f)
    plot_mag_histogram(ts, f)

summary = summary_table(ts, filters)
